# HARP-PCB — KiCad Auto-Placement (Google Colab)

Upload an **unplaced, unrouted** `.kicad_pcb` board, run the open-source placer, and download a new KiCad board plus a geometry report and preview.

**Important:** This is prototype placement only, not autorouting or KiCad DRC. Always check the output in KiCad. **The AI key is optional and never required for placement**. If enabled, the AI gets only selected report metrics, not your PCB source.

In [ ]:
#@title 1. Optional IntelCAD API key (NOT required for placement)
from getpass import getpass
import os

USE_AI_REVIEW = False #@param {type:"boolean"}
AI_MODEL = "cc/claude-haiku-5-5" #@param ["cc/claude-haiku-5-5", "cc/claude-opus-5-5"]
AI_BASE_URL = "https://ai-router.intelcad.ai/v1"
AI_API_KEY = ""
if USE_AI_REVIEW:
    try:
        from google.colab import userdata
        AI_API_KEY = userdata.get("INTELCAD_API_KEY") or ""
    except Exception:
        pass
    if not AI_API_KEY:
        AI_API_KEY = getpass("Optional IntelCAD API key (hidden input): ")
    if not AI_API_KEY:
        raise ValueError("AI review was enabled but no API key was provided. Disable AI review to continue without a key.")
print("AI review:", "enabled (report only)" if USE_AI_REVIEW else "disabled; local deterministic placement")

In [ ]:
#@title 2. Clone repository and install placement engine
from pathlib import Path
import subprocess, sys
REPO_URL = "https://github.com/amirhossein-alishahi/PCB_placement.git"
REPO_DIR = Path("/content/PCB_placement")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[preview]"], check=True)
print("Repository ready:", REPO_DIR)

In [ ]:
#@title 3. Upload an overlapping, unrouted .kicad_pcb file
from google.colab import files
from pathlib import Path
print("Choose ONE .kicad_pcb file, with overlapping/unplaced footprints.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please upload exactly one KiCad PCB file.")
filename = next(iter(uploaded))
if not filename.lower().endswith(".kicad_pcb"):
    raise ValueError("Expected a .kicad_pcb file")
INPUT_BOARD = (Path.cwd() / Path(filename).name).resolve()
print("Input:", INPUT_BOARD.name, "(", INPUT_BOARD.stat().st_size, "bytes)")

In [ ]:
#@title 4. Auto-place + create board outline + verify
import json
from IPython.display import display, Image
ITERATIONS = 3000 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
CLEARANCE_MM = 0.65 #@param {type:"number"}
EDGE_MARGIN_MM = 2.0 #@param {type:"number"}
OUTPUT_DIR = Path("/content/harp_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BOARD = OUTPUT_DIR / "placed.kicad_pcb"
OUTPUT_PREVIEW = OUTPUT_DIR / "placement_preview.png"
OUTPUT_REPORT = OUTPUT_DIR / "placement_report.json"
cmd = [sys.executable, str(REPO_DIR / "harp_pcb.py"), str(INPUT_BOARD),
       "-o", str(OUTPUT_BOARD), "--preview", str(OUTPUT_PREVIEW),
       "--report", str(OUTPUT_REPORT), "--iterations", str(ITERATIONS),
       "--seed", str(SEED), "--clearance", str(CLEARANCE_MM),
       "--edge-margin", str(EDGE_MARGIN_MM)]
subprocess.run(cmd, check=True)
report = json.loads(OUTPUT_REPORT.read_text(encoding="utf-8"))
print("\nBoard: %.2f x %.2f mm | footprints: %d | minimum bbox gap: %.3f mm" %
      (report["board_width_mm"], report["board_height_mm"],
       report["footprints"], report["min_bounding_box_gap_mm"]))
display(Image(filename=str(OUTPUT_PREVIEW)))
print("WARNING: Placement-only. The PCB is NOT routed or verified with official KiCad DRC.")

In [ ]:
#@title 5. Optional AI review of placement REPORT (no PCB data transmitted)
# This step does not change placement. The API is assumed OpenAI-compatible;
# unsupported model/router configurations are reported instead of blocking output.
if not USE_AI_REVIEW:
    print("Skipped AI review (not required for auto-placement).")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai>=1.0"], check=True)
    from openai import OpenAI
    metrics_only = {k: report.get(k) for k in (
        "footprints", "pads", "nets", "board_width_mm", "board_height_mm",
        "board_area_mm2", "min_bounding_box_gap_mm",
        "perimeter_and_clearance_pass", "proxy_wirelength_weighted_mm"
    )}
    metrics_only["courtyard_missing_count"] = len(report.get("courtyard_missing", []))
    client = OpenAI(api_key=AI_API_KEY, base_url=AI_BASE_URL, timeout=60.0)
    try:
        result = client.chat.completions.create(
            model=AI_MODEL,
            messages=[
                {"role": "system", "content": "You are an EDA reviewer. Analyze only the reported placement metrics. Do not claim to have run routing, KiCad DRC, or manufacturing validation. Give concise actionable warnings and next checks."},
                {"role": "user", "content": json.dumps(metrics_only, ensure_ascii=False)},
            ],
            temperature=0,
        )
        review = result.choices[0].message.content or "(empty response)"
        (OUTPUT_DIR / "optional_ai_review.txt").write_text(review, encoding="utf-8")
        print(review)
    except Exception as exc:
        print("Optional AI review failed; local placed PCB is unaffected:", type(exc).__name__, str(exc)[:250])

In [ ]:
#@title 6. Download placement output ZIP
import zipfile
from google.colab import files
archive = Path("/content/HARP_PCB_placement_result.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUTPUT_DIR.iterdir()):
        if file.is_file():
            z.write(file, arcname=file.name)
print("Files in archive:", [p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])
files.download(str(archive))
print("Inspect placed.kicad_pcb in KiCad and run kicad-cli pcb drc on a KiCad-equipped machine.")